# ESRGAN 2x / 23-block / Simple Degradation 训练（Colab 一体化）

本 Notebook 包含从 **环境初始化 → 退化数据生成 → 模型训练 → 推理评估** 的完整流程。  
在 Colab 中**从上到下运行所有 cell** 即可完成一次完整实验。

### 存储方案
| 位置 | 内容 |
|------|------|
| UCSD OneDrive | `dataset.zip`（含 `dataset/` + `pretrained_models/`） |
| UCSD Google Drive (5 GB) | `saved_models/`, `results/`（持久化） |
| GitHub `main` | 代码 + `APISR_tools/` |
| Colab 本地 SSD | 下载解压后的 dataset（临时） |

### 三阶段训练策略
| 阶段 | 说明 |
|------|------|
| Phase 1: G warmup | 只训练 Generator（L1 loss），稳定预训练权重 |
| Phase 2: D warmup | 冻结 Generator，只训练 Discriminator，让 D 追上 G |
| Phase 3: Full GAN | G + D 对抗训练（L1 + Perceptual + GAN） |

In [ ]:
# ============================================================
# 0. 全局配置
# ============================================================

GITHUB_REPO      = 'https://github.com/liqiqinaOH7/AWM.git'
BRANCH           = 'main'
ONEDRIVE_ZIP_URL = 'https://ucsdcloud-my.sharepoint.com/:u:/g/personal/xil326_ucsd_edu/IQDFdjcJu1AAQ67p-ln0IB85AdUSjend_ov_nfx_A6DNlX0?download=1'
DRIVE_ROOT       = '/content/drive/MyDrive/AWM'
PROJECT_DIR      = '/content/AWM'

In [ ]:
# ============================================================
# 1. 挂载 Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
for d in ['saved_models', 'results']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)
print(f'Drive 就绪: {DRIVE_ROOT}')

In [ ]:
# ============================================================
# 2. 从 OneDrive 下载 dataset.zip（含 dataset/ + pretrained_models/）
# ============================================================
import os, time

zip_path = '/content/dataset.zip'
marker   = '/content/dataset/highres/original'

if os.path.isdir(marker):
    print('dataset 已存在，跳过下载。')
else:
    print('正在从 OneDrive 下载 dataset.zip（~3.4 GB）...')
    t0 = time.time()
    !wget -q --show-progress -O "{zip_path}" "{ONEDRIVE_ZIP_URL}"
    elapsed = time.time() - t0
    if os.path.isfile(zip_path) and os.path.getsize(zip_path) > 1_000_000:
        print(f'下载完成: {os.path.getsize(zip_path)/1e9:.2f} GB, {elapsed:.0f}s，解压中...')
        !unzip -q -o "{zip_path}" -d /content/
        os.remove(zip_path)
        print('解压完成。')
    else:
        print('⚠ 下载可能失败，请检查 OneDrive 链接权限和 URL。')

In [ ]:
# ============================================================
# 2b. 备用：如果 wget 下载失败，手动上传 dataset.zip 后运行此 cell
# ============================================================
# from google.colab import files
# uploaded = files.upload()
# import os
# if os.path.isfile('/content/dataset.zip'):
#     !unzip -q -o /content/dataset.zip -d /content/
#     os.remove('/content/dataset.zip')
#     print('解压完成。')

In [ ]:
# ============================================================
# 3. 拉取 GitHub 代码 + 创建软链接
# ============================================================
import os

if os.path.isdir(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f'工作目录: {os.getcwd()}')

links = {
    'dataset':           '/content/dataset',
    'pretrained_models': '/content/pretrained_models',
    'saved_models':      f'{DRIVE_ROOT}/saved_models',
    'results':           f'{DRIVE_ROOT}/results',
}
for name, target in links.items():
    lp = os.path.join(PROJECT_DIR, name)
    if os.path.islink(lp):
        if os.readlink(lp) == target: continue
        os.unlink(lp)
    elif os.path.isdir(lp): continue
    os.symlink(target, lp)
    print(f'  链接: {name} → {target}')

!pip install -q pyiqa tqdm
print('\n环境初始化完成。')

In [ ]:
# ============================================================
# 4. 生成简化版 2x LR 数据集（仅 bicubic 下采样，无滤波）
#    如果已存在则跳过
# ============================================================
import os, glob, cv2
from tqdm import tqdm

hr_dir = 'dataset/highres/original'
lr_dir = 'dataset/lowres_2x_simple/original'
scale  = 2

os.makedirs(lr_dir, exist_ok=True)
hr_image_paths = sorted(glob.glob(os.path.join(hr_dir, '*.[jp][pn]*g')))
existing_lr = len(os.listdir(lr_dir)) if os.path.isdir(lr_dir) else 0

if existing_lr >= len(hr_image_paths) and existing_lr > 0:
    print(f'lowres_2x_simple 已有 {existing_lr} 张，跳过生成。')
else:
    print(f'正在生成简化版 2x LR（{len(hr_image_paths)} 张 HR）...')
    count = 0
    for hr_path in tqdm(hr_image_paths, desc='2x bicubic'):
        img = cv2.imread(hr_path)
        if img is None: continue
        h, w = img.shape[:2]
        img_lr = cv2.resize(img, (w // scale, h // scale), interpolation=cv2.INTER_CUBIC)
        fn = os.path.basename(hr_path)
        if fn.lower().endswith('.png'): fn = fn[:-4] + '.jpg'
        cv2.imwrite(os.path.join(lr_dir, fn), img_lr, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
        count += 1
    print(f'完成: {count} 张 LR 保存到 {lr_dir}')

In [ ]:
# ============================================================
# 5. 验证数据集
# ============================================================
import glob, os
import torch, cv2

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} \u2014 {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

hr_count = len(glob.glob('dataset/highres/original/*.[jp][pn]*g'))
lr_count = len(glob.glob('dataset/lowres_2x_simple/original/*.[jp][pn]*g'))
print(f'HR: {hr_count} 张 | LR (2x simple): {lr_count} 张')

pt = glob.glob('pretrained_models/*.pth')
print(f'预训练权重: {[os.path.basename(f) for f in pt]}')

assert hr_count > 0, '未检测到 HR 图片'
assert lr_count > 0, '未检测到 LR 图片'
print('\n\u2705 数据就绪！')

---
## 训练部分

In [ ]:
# ============================================================
# 6. Import
# ============================================================
import os, sys, glob, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

apisr_tools_path = os.path.abspath('APISR_tools')
if apisr_tools_path not in sys.path:
    sys.path.append(apisr_tools_path)

from architecture.rrdb import RRDBNet
from architecture.discriminator import UNetDiscriminatorSN
from loss.perceptual_loss import PerceptualLoss
from loss.anime_perceptual_loss import Anime_PerceptualLoss
from loss.gan_loss import GANLoss

print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# ============================================================
# 7. 超参数配置
# ============================================================

UPSCALE_MODE = '2x'
SCALE = 2

LR_DIR = 'dataset/lowres_2x_simple/original'
HR_DIR = 'dataset/highres/original'
PRETRAINED_DIR = 'pretrained_models'

BATCH_SIZE    = 8
PATCH_SIZE    = 256       # HR 裁剪尺寸，LR 对应 128
NUM_EPOCHS    = 150
LEARNING_RATE = 1e-5
SAVE_FREQ     = 10

NUM_BLOCK = 23

LR_STEP_SIZE = 30
LR_GAMMA     = 0.5
LR_MIN       = 1e-6

CHECKPOINT_NAME = 'ESRGAN_2x_23block_simple'
MONITOR_IMAGE   = '372812'

# --- 三阶段训练策略 ---
WARMUP_EPOCHS    = 5      # Phase 1: G warmup (L1 only)
D_WARMUP_EPOCHS  = 5      # Phase 2: D warmup (冻结 G, 只训 D)

WEIGHT_PIXEL     = 10.0
WEIGHT_VGG       = 0.05
WEIGHT_DANBOORU  = 0.025
WEIGHT_GAN       = 1.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'\n训练阶段规划:')
print(f'  Phase 1 (Epoch 1-{WARMUP_EPOCHS}): G warmup \u2014 L1 only')
print(f'  Phase 2 (Epoch {WARMUP_EPOCHS+1}-{WARMUP_EPOCHS+D_WARMUP_EPOCHS}): D warmup \u2014 freeze G, train D only')
print(f'  Phase 3 (Epoch {WARMUP_EPOCHS+D_WARMUP_EPOCHS+1}-{NUM_EPOCHS}): Full GAN')

In [ ]:
# ============================================================
# 8. 数据集
# ============================================================

class AnimeSRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, patch_size=256, scale=2):
        self.lr_paths = sorted(glob.glob(os.path.join(lr_dir, '*.*')))
        self.hr_paths = sorted(glob.glob(os.path.join(hr_dir, '*.*')))
        self.patch_size = patch_size
        self.scale = scale
        print(f'LR: {len(self.lr_paths)} | HR: {len(self.hr_paths)}')
        assert len(self.lr_paths) == len(self.hr_paths), 'LR / HR 数量不匹配！'

    def __len__(self):
        return len(self.hr_paths)

    def __getitem__(self, idx):
        hr_img = cv2.imread(self.hr_paths[idx])
        lr_img = cv2.imread(self.lr_paths[idx])
        if hr_img is None or lr_img is None:
            raise ValueError(f'无法读取: {self.hr_paths[idx]} / {self.lr_paths[idx]}')
        hr_img = cv2.cvtColor(hr_img, cv2.COLOR_BGR2RGB)
        lr_img = cv2.cvtColor(lr_img, cv2.COLOR_BGR2RGB)

        h_hr, w_hr, _ = hr_img.shape
        h_lr, w_lr, _ = lr_img.shape
        lps = self.patch_size // self.scale

        if h_hr < self.patch_size or w_hr < self.patch_size:
            hr_img = cv2.resize(hr_img, (max(w_hr, self.patch_size), max(h_hr, self.patch_size)))
            lr_img = cv2.resize(lr_img, (max(w_lr, lps), max(h_lr, lps)))
            h_hr, w_hr, _ = hr_img.shape
            h_lr, w_lr, _ = lr_img.shape

        x_lr = random.randint(0, w_lr - lps)
        y_lr = random.randint(0, h_lr - lps)
        x_hr, y_hr = x_lr * self.scale, y_lr * self.scale

        hr_patch = hr_img[y_hr:y_hr+self.patch_size, x_hr:x_hr+self.patch_size, :]
        lr_patch = lr_img[y_lr:y_lr+lps, x_lr:x_lr+lps, :]

        if hr_patch.size == 0 or lr_patch.size == 0:
            raise ValueError(f'裁剪失败: hr={hr_img.shape}, lr={lr_img.shape}, ps={self.patch_size}, lps={lps}')

        if random.random() < 0.5:
            hr_patch = cv2.flip(hr_patch, 1); lr_patch = cv2.flip(lr_patch, 1)
        if random.random() < 0.5:
            hr_patch = cv2.flip(hr_patch, 0); lr_patch = cv2.flip(lr_patch, 0)
        if random.random() < 0.5:
            hr_patch = hr_patch.transpose(1,0,2); lr_patch = lr_patch.transpose(1,0,2)

        hr_t = torch.from_numpy(hr_patch.transpose(2,0,1)).float() / 255.0
        lr_t = torch.from_numpy(lr_patch.transpose(2,0,1)).float() / 255.0
        return {'lr': lr_t, 'hr': hr_t}

train_dataset = AnimeSRDataset(LR_DIR, HR_DIR, patch_size=PATCH_SIZE, scale=SCALE)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
print(f'DataLoader 就绪: {len(train_dataset)} samples, {len(train_loader)} batches/epoch')

In [ ]:
# ============================================================
# 9. 模型 + 加载预训练权重
# ============================================================

generator     = RRDBNet(3, 3, scale=SCALE, num_block=NUM_BLOCK).to(device)
discriminator = UNetDiscriminatorSN(3).to(device)

pattern = os.path.join(PRETRAINED_DIR, '*2x*.pth')
pretrained_paths = sorted(glob.glob(pattern))
if not pretrained_paths:
    pretrained_paths = sorted(glob.glob(os.path.join(PRETRAINED_DIR, '*.pth')))

if pretrained_paths:
    pp = pretrained_paths[0]
    print(f'加载预训练: {pp}')
    ckpt = torch.load(pp, map_location=device)
    if 'params_ema' in ckpt:       state = ckpt['params_ema']
    elif 'model_state_dict' in ckpt: state = ckpt['model_state_dict']
    else:                            state = ckpt
    model_state = generator.state_dict()
    compatible = {k: v for k, v in state.items() if k in model_state and model_state[k].shape == v.shape}
    generator.load_state_dict(compatible, strict=False)
    print(f'匹配 {len(compatible)} / {len(state)} 参数 (跳过 {len(state)-len(compatible)})')
else:
    print('\u26a0 未找到预训练权重，将从头训练。')

In [ ]:
# ============================================================
# 10. 损失函数 + 优化器 + 调度器
# ============================================================

cri_pix = nn.L1Loss().to(device)

cri_vgg = PerceptualLoss(
    layer_weights={'conv1_2': 0.1, 'conv2_2': 0.1, 'conv3_4': 1, 'conv4_4': 1, 'conv5_4': 1},
    vgg_type='vgg19',
    perceptual_weight=WEIGHT_VGG
).to(device)

cri_danbooru = Anime_PerceptualLoss(
    layer_weights={'0': 0.1, '4_2_conv3': 20, '5_3_conv3': 25, '6_5_conv3': 1, '7_2_conv3': 1},
    perceptual_weight=WEIGHT_DANBOORU
).to(device)

cri_gan = GANLoss(gan_type='vanilla', loss_weight=WEIGHT_GAN).to(device)

optimizer_g = optim.Adam(generator.parameters(),     lr=LEARNING_RATE, betas=(0.9, 0.999))
optimizer_d = optim.Adam(discriminator.parameters(),  lr=LEARNING_RATE, betas=(0.9, 0.999))

scheduler_g = optim.lr_scheduler.StepLR(optimizer_g, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)
scheduler_d = optim.lr_scheduler.StepLR(optimizer_d, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

scaler = torch.amp.GradScaler('cuda')

print('损失函数、优化器、调度器初始化完成。')

In [ ]:
# ============================================================
# 10b. 监控推理函数
# ============================================================

monitor_lr_path, monitor_hr_path = None, None
for ext in ['jpg', 'jpeg', 'png', 'JPG', 'JPEG', 'PNG']:
    for p in glob.glob(f'{LR_DIR}/{MONITOR_IMAGE}.{ext}'):
        monitor_lr_path = p
    for p in glob.glob(f'{HR_DIR}/{MONITOR_IMAGE}.{ext}'):
        monitor_hr_path = p

if monitor_lr_path and monitor_hr_path:
    print(f'监控图片: LR={monitor_lr_path}')
    print(f'         HR={monitor_hr_path}')
else:
    print(f'\u26a0 未找到监控图片 {MONITOR_IMAGE}，将跳过可视化监控。')


def monitor_inference(gen, disc, epoch, dev):
    if not monitor_lr_path or not monitor_hr_path:
        return None

    gen.eval(); disc.eval()

    lr_img = cv2.imread(monitor_lr_path)
    hr_img = cv2.imread(monitor_hr_path)
    if lr_img is None or hr_img is None:
        gen.train(); disc.train()
        return None
    lr_img = cv2.cvtColor(lr_img, cv2.COLOR_BGR2RGB)
    hr_img = cv2.cvtColor(hr_img, cv2.COLOR_BGR2RGB)

    h_lr, w_lr = lr_img.shape[:2]
    crop_sz = min(128, h_lr, w_lr)
    cy, cx = h_lr // 2, w_lr // 2
    half = crop_sz // 2
    lr_crop = lr_img[cy-half:cy+half, cx-half:cx+half]
    hr_crop = hr_img[(cy-half)*SCALE:(cy+half)*SCALE,
                     (cx-half)*SCALE:(cx+half)*SCALE]

    # 2x RRDB 需要输入尺寸能被 4 整除（pixel_unshuffle）
    ch, cw = lr_crop.shape[:2]
    pad_h = (4 - ch % 4) % 4
    pad_w = (4 - cw % 4) % 4
    if pad_h > 0 or pad_w > 0:
        lr_crop = cv2.copyMakeBorder(lr_crop, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)

    lr_t = torch.from_numpy(lr_crop.transpose(2,0,1)).float().unsqueeze(0).to(dev) / 255.0
    hr_t = torch.from_numpy(hr_crop.transpose(2,0,1)).float().unsqueeze(0).to(dev) / 255.0

    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            sr_t = gen(lr_t)
            d_real = torch.sigmoid(disc(hr_t)).mean().item()
            d_fake = torch.sigmoid(disc(sr_t)).mean().item()

    sr_np = sr_t.squeeze(0).float().cpu().clamp(0, 1).numpy().transpose(1,2,0)
    # 裁掉 padding 对应的 SR 边界
    sr_np = sr_np[:ch*SCALE, :cw*SCALE, :]
    lr_up = cv2.resize(lr_crop[:ch, :cw], (cw*SCALE, ch*SCALE),
                       interpolation=cv2.INTER_CUBIC).astype(np.float32) / 255.0
    hr_np = hr_crop.astype(np.float32) / 255.0

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(lr_up);  axes[0].set_title('LR (bicubic upscale)')
    axes[1].imshow(sr_np);  axes[1].set_title(f'SR (Epoch {epoch+1})')
    axes[2].imshow(hr_np);  axes[2].set_title('HR (Ground Truth)')
    for ax in axes: ax.axis('off')
    fig.suptitle(f'Epoch {epoch+1}  |  D(real)={d_real:.4f}  D(fake)={d_fake:.4f}', fontsize=14)
    plt.tight_layout()
    fig.savefig(f'saved_models/monitor_2x_epoch_{epoch+1:03d}.png', dpi=100, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    gen.train(); disc.train()
    return d_real, d_fake

In [ ]:
# ============================================================
# 11. 训练主循环（三阶段：G warmup → D warmup → Full GAN）
# ============================================================

os.makedirs('saved_models', exist_ok=True)
assert os.path.isdir(LR_DIR), f'LR 路径不存在: {LR_DIR}'
assert os.path.isdir(HR_DIR), f'HR 路径不存在: {HR_DIR}'

print('开始训练...')
best_loss = float('inf')
history_loss_g, history_loss_d = [], []
history_d_real, history_d_fake = [], []

for epoch in range(NUM_EPOCHS):
    generator.train()
    discriminator.train()
    epoch_loss_g, epoch_loss_d = 0.0, 0.0

    if epoch > 0:
        scheduler_g.step()
        scheduler_d.step()
        for opt in [optimizer_g, optimizer_d]:
            for pg in opt.param_groups:
                pg['lr'] = max(pg['lr'], LR_MIN)

    is_g_warmup = epoch < WARMUP_EPOCHS
    is_d_warmup = WARMUP_EPOCHS <= epoch < (WARMUP_EPOCHS + D_WARMUP_EPOCHS)
    is_full_gan = epoch >= (WARMUP_EPOCHS + D_WARMUP_EPOCHS)

    if is_g_warmup:   phase = 'Phase 1: G warmup (L1 only)'
    elif is_d_warmup: phase = 'Phase 2: D warmup (freeze G, train D only)'
    else:             phase = 'Phase 3: Full GAN'
    print(f'--- Epoch {epoch+1}/{NUM_EPOCHS}: {phase} ---')

    pbar = tqdm(train_loader, desc=f'Epoch [{epoch+1}/{NUM_EPOCHS}]')
    for batch in pbar:
        imgs_lr = batch['lr'].to(device)
        imgs_hr = batch['hr'].to(device)
        l_pix = l_percep = l_gan_g = torch.tensor(0.0, device=device)
        loss_g = loss_d = torch.tensor(0.0, device=device)

        if is_d_warmup:
            with torch.no_grad():
                gen_hr = generator(imgs_lr)
            for p in discriminator.parameters(): p.requires_grad = True
            optimizer_d.zero_grad()
            with torch.amp.autocast('cuda'):
                l_d_real = cri_gan(discriminator(imgs_hr),         True,  is_disc=True)
                l_d_fake = cri_gan(discriminator(gen_hr.detach()), False, is_disc=True)
                loss_d = l_d_real + l_d_fake
            scaler.scale(loss_d).backward()
            scaler.step(optimizer_d)
            scaler.update()
            epoch_loss_d += loss_d.item()

        else:
            optimizer_g.zero_grad()
            for p in discriminator.parameters(): p.requires_grad = False
            with torch.amp.autocast('cuda'):
                gen_hr = generator(imgs_lr)
                l_pix = cri_pix(gen_hr, imgs_hr) * WEIGHT_PIXEL
                if is_g_warmup:
                    loss_g = l_pix
                else:
                    l_percep = cri_vgg(gen_hr, imgs_hr) + cri_danbooru(gen_hr, imgs_hr)
                    l_gan_g  = cri_gan(discriminator(gen_hr), True, is_disc=False)
                    loss_g = l_pix + l_percep + l_gan_g
            scaler.scale(loss_g).backward()
            scaler.step(optimizer_g)
            scaler.update()
            epoch_loss_g += loss_g.item()

            if is_full_gan:
                for p in discriminator.parameters(): p.requires_grad = True
                optimizer_d.zero_grad()
                with torch.amp.autocast('cuda'):
                    l_d_real = cri_gan(discriminator(imgs_hr),         True,  is_disc=True)
                    l_d_fake = cri_gan(discriminator(gen_hr.detach()), False, is_disc=True)
                    loss_d = l_d_real + l_d_fake
                scaler.scale(loss_d).backward()
                scaler.step(optimizer_d)
                scaler.update()
                epoch_loss_d += loss_d.item()

        pbar.set_postfix(G=f'{loss_g.item():.4f}', D=f'{loss_d.item():.4f}',
                         L1=f'{l_pix.item():.4f}', P=f'{l_percep.item():.4f}', GAN=f'{l_gan_g.item():.4f}')

    avg_g = epoch_loss_g / len(train_loader) if not is_d_warmup else 0.0
    avg_d = epoch_loss_d / len(train_loader) if not is_g_warmup else 0.0
    history_loss_g.append(avg_g)
    history_loss_d.append(avg_d)
    cur_lr = optimizer_g.param_groups[0]['lr']
    print(f'Epoch {epoch+1} | G: {avg_g:.4f} | D: {avg_d:.4f} | LR: {cur_lr:.6f}')

    d_scores = monitor_inference(generator, discriminator, epoch, device)
    if d_scores:
        history_d_real.append(d_scores[0])
        history_d_fake.append(d_scores[1])

    if is_full_gan and avg_g < best_loss:
        best_loss = avg_g
        bp = f'saved_models/{CHECKPOINT_NAME}_best.pth'
        torch.save({'epoch': epoch+1, 'model_state_dict': generator.state_dict(),
                     'optimizer_state_dict': optimizer_g.state_dict(), 'loss': avg_g}, bp)
        print(f'  \u2192 Best model saved: {bp}')

    if (epoch+1) % SAVE_FREQ == 0 or (epoch+1) == NUM_EPOCHS:
        sp = f'saved_models/{CHECKPOINT_NAME}_epoch_{epoch+1}.pth'
        torch.save({'epoch': epoch+1, 'model_state_dict': generator.state_dict(),
                     'optimizer_state_dict': optimizer_g.state_dict(), 'loss': avg_g}, sp)
        print(f'  \u2192 Checkpoint saved: {sp}')

print('\n训练完成！')

In [ ]:
# ============================================================
# 12. 绘制损失曲线 + D 分数趋势
# ============================================================
if history_loss_g:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    epochs_range = range(1, len(history_loss_g)+1)

    ax1.plot(epochs_range, history_loss_g, label='Generator Loss')
    ax1.plot(epochs_range, history_loss_d, label='Discriminator Loss')
    ax1.axvline(x=WARMUP_EPOCHS, color='orange', linestyle='--', alpha=0.6, label='G warmup end')
    ax1.axvline(x=WARMUP_EPOCHS+D_WARMUP_EPOCHS, color='red', linestyle='--', alpha=0.6, label='D warmup end')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss'); ax1.legend(); ax1.grid(True)

    if history_d_real:
        dr = range(1, len(history_d_real)+1)
        ax2.plot(dr, history_d_real, label='D(real)', color='green')
        ax2.plot(dr, history_d_fake, label='D(fake)', color='red')
        ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='D=0.5 (balanced)')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('D score (sigmoid)')
        ax2.set_title('Discriminator Predictions on Monitor Image')
        ax2.legend(); ax2.grid(True)
    else:
        ax2.text(0.5, 0.5, 'No monitor data', ha='center', va='center', transform=ax2.transAxes)

    plt.suptitle('2x / 23-block / simple degradation', fontsize=14)
    plt.tight_layout()
    plt.savefig('saved_models/loss_curve_2x_23block_simple.png')
    plt.show()
else:
    print('无训练记录。')

---
## 推理 + 评估 + 持久化保存

In [ ]:
# ============================================================
# 13. 确认 best 模型已持久化到 Google Drive
# ============================================================
import os

best_model_path = f'saved_models/{CHECKPOINT_NAME}_best.pth'
drive_model_dir = f'{DRIVE_ROOT}/saved_models'

if os.path.isfile(best_model_path):
    real_path = os.path.realpath(best_model_path)
    print(f'Best 模型: {best_model_path}')
    print(f'  实际路径: {real_path}')
    print(f'  文件大小: {os.path.getsize(best_model_path)/1e6:.1f} MB')
    if real_path.startswith('/content/drive'):
        print('  \u2705 已在 Google Drive 上，无需复制。')
    else:
        import shutil
        dst = os.path.join(drive_model_dir, os.path.basename(best_model_path))
        shutil.copy2(best_model_path, dst)
        print(f'  已复制到: {dst}')
else:
    import glob as _g
    ckpts = sorted(_g.glob(f'saved_models/{CHECKPOINT_NAME}_epoch_*.pth'))
    if ckpts:
        best_model_path = ckpts[-1]
        print(f'未找到 best，使用最后 checkpoint: {best_model_path}')
    else:
        print('\u26a0 未找到任何 checkpoint')

In [ ]:
# ============================================================
# 14. 加载 best 模型进行推理
# ============================================================

OUTPUT_DIR = f'results/ESRGAN_{CHECKPOINT_NAME}_inference'
os.makedirs(OUTPUT_DIR, exist_ok=True)

infer_model = RRDBNet(3, 3, scale=SCALE, num_block=NUM_BLOCK).to(device)
print(f'加载权重: {best_model_path}')
ckpt = torch.load(best_model_path, map_location=device)
if 'model_state_dict' in ckpt:
    infer_model.load_state_dict(ckpt['model_state_dict'])
    print(f'  Epoch {ckpt.get("epoch", "?")} | Loss {ckpt.get("loss", "?")}')
elif 'params_ema' in ckpt:
    infer_model.load_state_dict(ckpt['params_ema'])
else:
    infer_model.load_state_dict(ckpt)
infer_model.eval()

def tensor2img(tensor):
    img = tensor.squeeze(0).float().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    img = np.clip(img * 255.0, 0, 255).astype(np.uint8)
    return img

def infer_and_save(lr_path, save_dir, scale=2):
    lr_img = cv2.imread(lr_path)
    if lr_img is None:
        return None
    lr_img_rgb = cv2.cvtColor(lr_img, cv2.COLOR_BGR2RGB)

    orig_h, orig_w = lr_img_rgb.shape[:2]
    pad_h = (4 - orig_h % 4) % 4
    pad_w = (4 - orig_w % 4) % 4
    lr_padded = lr_img_rgb
    if pad_h > 0 or pad_w > 0:
        lr_padded = cv2.copyMakeBorder(lr_img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)

    lr_tensor = torch.from_numpy(lr_padded.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            _, _, h, w = lr_tensor.shape
            if h * w > 512 * 512:
                h_half, w_half = h // 2, w // 2
                sr_tensor = torch.zeros((1, 3, h * scale, w * scale), device=device)
                sr_tensor[:, :, :h_half*scale, :w_half*scale] = infer_model(lr_tensor[:, :, :h_half, :w_half])
                sr_tensor[:, :, :h_half*scale, w_half*scale:] = infer_model(lr_tensor[:, :, :h_half, w_half:])
                sr_tensor[:, :, h_half*scale:, :w_half*scale] = infer_model(lr_tensor[:, :, h_half:, :w_half])
                sr_tensor[:, :, h_half*scale:, w_half*scale:] = infer_model(lr_tensor[:, :, h_half:, w_half:])
            else:
                sr_tensor = infer_model(lr_tensor)

    sr_img = tensor2img(sr_tensor)
    if pad_h > 0 or pad_w > 0:
        sr_img = sr_img[:orig_h * scale, :orig_w * scale, :]

    os.makedirs(save_dir, exist_ok=True)
    fn = os.path.basename(lr_path)
    cv2.imwrite(os.path.join(save_dir, fn), cv2.cvtColor(sr_img, cv2.COLOR_RGB2BGR))
    return sr_img

lr_paths = sorted(glob.glob(os.path.join(LR_DIR, '*.[jp][pn]*g')))
print(f'开始推理 {len(lr_paths)} 张 LR 图像 → {OUTPUT_DIR}')
for lp in tqdm(lr_paths, desc='Inference'):
    infer_and_save(lp, OUTPUT_DIR, scale=SCALE)
print(f'推理完成，共 {len(os.listdir(OUTPUT_DIR))} 张 SR 图像。')

In [ ]:
# ============================================================
# 15. 推理可视化（随机 3 张 LR / SR / HR 对比）
# ============================================================

sample_lr = random.sample(lr_paths, min(3, len(lr_paths)))
for lp in sample_lr:
    fn = os.path.basename(lp)
    sr_path = os.path.join(OUTPUT_DIR, fn)
    hr_candidates = glob.glob(os.path.join(HR_DIR, os.path.splitext(fn)[0] + '.*'))

    sr_img = cv2.cvtColor(cv2.imread(sr_path), cv2.COLOR_BGR2RGB) if os.path.isfile(sr_path) else None
    lr_img = cv2.cvtColor(cv2.imread(lp), cv2.COLOR_BGR2RGB)
    hr_img = cv2.cvtColor(cv2.imread(hr_candidates[0]), cv2.COLOR_BGR2RGB) if hr_candidates else None

    cols = 2 + (1 if hr_img is not None else 0)
    fig, axes = plt.subplots(1, cols, figsize=(6*cols, 6))
    h_target = sr_img.shape[0] if sr_img is not None else lr_img.shape[0] * SCALE
    w_target = sr_img.shape[1] if sr_img is not None else lr_img.shape[1] * SCALE

    lr_up = cv2.resize(lr_img, (w_target, h_target), interpolation=cv2.INTER_CUBIC)
    axes[0].imshow(lr_up); axes[0].set_title('LR (bicubic)'); axes[0].axis('off')
    if sr_img is not None:
        axes[1].imshow(sr_img); axes[1].set_title('SR (model)'); axes[1].axis('off')
    if hr_img is not None:
        axes[cols-1].imshow(hr_img); axes[cols-1].set_title('HR (GT)'); axes[cols-1].axis('off')
    fig.suptitle(fn, fontsize=12)
    plt.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
# ============================================================
# 16. 图像质量评估 (NIQE / MANIQA / CLIPIQA) + 结果持久化
# ============================================================
import pyiqa, json
from datetime import datetime

print('初始化评估指标（首次运行会自动下载权重）...')
niqe_metric    = pyiqa.create_metric('niqe',    device=device)
maniqa_metric  = pyiqa.create_metric('maniqa',  device=device)
clipiqa_metric = pyiqa.create_metric('clipiqa', device=device)

sr_paths = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.*')))
niqe_scores, maniqa_scores, clipiqa_scores = [], [], []

print(f'评估 {len(sr_paths)} 张 SR 图像...')
for sp in tqdm(sr_paths, desc='Evaluating'):
    niqe_scores.append(niqe_metric(sp).item())
    maniqa_scores.append(maniqa_metric(sp).item())
    clipiqa_scores.append(clipiqa_metric(sp).item())

avg_niqe    = np.mean(niqe_scores)
avg_maniqa  = np.mean(maniqa_scores)
avg_clipiqa = np.mean(clipiqa_scores)

print('\n' + '='*50)
print(f'  {CHECKPOINT_NAME} 评估结果')
print('='*50)
print(f'  Average NIQE   : {avg_niqe:.4f} (越低越好)')
print(f'  Average MANIQA : {avg_maniqa:.4f} (越高越好)')
print(f'  Average CLIPIQA: {avg_clipiqa:.4f} (越高越好)')
print('='*50)

eval_result = {
    'model': CHECKPOINT_NAME,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'num_images': len(sr_paths),
    'scale': SCALE,
    'num_block': NUM_BLOCK,
    'epochs': NUM_EPOCHS,
    'best_model': os.path.basename(best_model_path),
    'metrics': {
        'NIQE': round(avg_niqe, 4),
        'MANIQA': round(avg_maniqa, 4),
        'CLIPIQA': round(avg_clipiqa, 4),
    },
    'per_image': {os.path.basename(sr_paths[i]): {
        'NIQE': round(niqe_scores[i], 4),
        'MANIQA': round(maniqa_scores[i], 4),
        'CLIPIQA': round(clipiqa_scores[i], 4),
    } for i in range(len(sr_paths))}
}

drive_results_dir = f'{DRIVE_ROOT}/results'
os.makedirs(drive_results_dir, exist_ok=True)

json_path = os.path.join(drive_results_dir, f'{CHECKPOINT_NAME}_eval.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(eval_result, f, indent=2, ensure_ascii=False)
print(f'\n评估结果 (JSON) 已保存: {json_path}')

txt_path = os.path.join(drive_results_dir, f'{CHECKPOINT_NAME}_eval.txt')
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write(f'{CHECKPOINT_NAME}\n')
    f.write(f'时间: {eval_result["timestamp"]}\n')
    f.write(f'模型: {eval_result["best_model"]}\n')
    f.write(f'Scale: {SCALE}x | Blocks: {NUM_BLOCK} | Epochs: {NUM_EPOCHS}\n')
    f.write('='*40 + '\n')
    f.write(f'Average NIQE   : {avg_niqe:.4f} (越低越好)\n')
    f.write(f'Average MANIQA : {avg_maniqa:.4f} (越高越好)\n')
    f.write(f'Average CLIPIQA: {avg_clipiqa:.4f} (越高越好)\n')
    f.write('='*40 + '\n')
print(f'评估结果 (TXT) 已保存: {txt_path}')
print(f'\n\u2705 所有结果已持久化到 Google Drive。')